# Revisión de los datos preparados y primera referencia

Ejecuta de arriba abajo. Los comentarios que empiezan por `#` explican las instrucciones.
Este notebook utiliza la ejecución validada que indica `data/processed/CURRENT.json`.
Las reglas de preparación y la partición viven en `process_data.py`, para que el equipo
utilice exactamente la misma definición. Los datos originales no se modifican.

**Qué predecimos:** ventas registradas la próxima semana para artículos vendidos en
la semana actual. No conocemos el stock ni la demanda que no llegó a convertirse
en ventas. Los artículos que no venden en la semana actual quedan fuera de este experimento.

## 1. Cargar la ejecución vigente

Si falta CURRENT, ejecuta primero `python src/data/process_data.py` desde el proyecto.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Buscamos la carpeta del proyecto desde la ubicación del notebook.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/data/process_data.py").is_file()
             and (p / "data/processed/CURRENT.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abre fashion-demand-ml y ejecuta primero python src/data/process_data.py")

carpeta_procesada = ROOT / "data/processed"
puntero = json.loads((carpeta_procesada / "CURRENT.json").read_text(encoding="utf-8"))
ejecucion = (carpeta_procesada / puntero["run_dir"]).resolve()
if not ejecucion.is_relative_to(carpeta_procesada.resolve()) or puntero.get("preview"):
    raise ValueError("La ejecución no es una publicación completa válida")
resumen = json.loads((ejecucion / "summary.json").read_text(encoding="utf-8"))
procedencia = json.loads((ejecucion / "provenance.json").read_text(encoding="utf-8"))
huellas = json.loads((ejecucion / "checksums.json").read_text(encoding="utf-8"))

# Verificamos que el archivo publicado no ha cambiado.
archivo = ejecucion / "article_week.csv.gz"
huella = hashlib.sha256()
with archivo.open("rb") as entrada:
    for bloque in iter(lambda: entrada.read(4 * 1024 * 1024), b""):
        huella.update(bloque)
if huella.hexdigest() != huellas[archivo.name]:
    raise ValueError("El archivo procesado ha cambiado después de publicarse")

# Los IDs conservan sus ceros y las semanas se interpretan como fechas.
datos_modelo = pd.read_csv(archivo,
    dtype={"article_id": "string", "product_code": "string"},
    parse_dates=["week", "target_week"])
print("Ejecución:", ejecucion.name)
print("Filas y columnas:", datos_modelo.shape)
display(datos_modelo.head())

Ejecución: 20261020T154621-9be94fc7
Filas y columnas: (1887139, 38)


,week,article_id,sales,min_price,max_price,online_sales,mean_price,online_share,lag_1_sales,lag_2_sales,lag_3_sales,lag_4_sales,lag_5_sales,lag_6_sales,lag_7_sales,lag_8_sales,lag_9_sales,lag_10_sales,lag_11_sales,lag_12_sales,target_next_week_sales,target_week,target_is_observed,history_is_observed,rolling_4w_sales_mean,rolling_12w_sales_mean,iso_year,iso_week,split,product_code,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,index_group_name,section_name,garment_group_name
0,2018-12-17,0108775015,227,0.006339,0.008458,178,0.008321,0.784141,300,337,298,680,244,312,373,434,259,360,315,547,229,2018-12-24,True,True,403.75,371.583333,2018,51,train,0108775,Vest top,Garment Upper body,Solid,Black,Dark,Ladieswear,Womens Everyday Basics,Jersey Basic
1,2018-12-17,0108775044,107,0.003034,0.008458,72,0.008233,0.672897,108,106,97,240,81,92,107,160,115,155,139,226,107,2018-12-24,True,True,137.75,135.500000,2018,51,train,0108775,Vest top,Garment Upper body,Solid,White,Light,Ladieswear,Womens Everyday Basics,Jersey Basic
2,2018-12-17,0108775051,1,0.005068,0.005068,1,0.005068,1.000000,2,3,1,4,2,4,5,19,24,44,45,38,0,2018-12-24,True,True,2.50,15.916667,2018,51,train,0108775,Vest top,Garment Upper body,Stripe,Off White,Dusty Light,Ladieswear,Womens Everyday Basics,Jersey Basic
3,2018-12-17,0110065001,18,0.021593,0.025407,7,0.024913,0.388889,14,31,14,64,25,30,25,50,23,45,37,56,22,2018-12-24,True,True,30.75,34.500000,2018,51,train,0110065,Bra,Underwear,Solid,Black,Dark,Ladieswear,Womens Lingerie,"Under-, Nightwear"
4,2018-12-17,0110065002,21,0.016932,0.025407,18,0.019374,0.857143,6,5,5,23,10,8,17,16,12,20,16,16,31,2018-12-24,True,True,9.75,12.833333,2018,51,train,0110065,Bra,Underwear,Solid,White,Light,Ladieswear,Womens Lingerie,"Under-, Nightwear"


## 2. Comprobar la calidad del resultado

Los precios se han agregado en float64. Usamos una tolerancia de 1e-12 al comparar
media, mínimo y máximo para admitir únicamente el redondeo de coma flotante.

In [2]:
# Comprobamos toda la tabla sin eliminar filas automáticamente.
display(pd.DataFrame({"tipo": datos_modelo.dtypes.astype(str),
                      "vacios": datos_modelo.isna().sum()}))
numeros = datos_modelo.select_dtypes(include="number")
tolerancia = 1e-12
incidencias = pd.Series({
    "Artículo-semana duplicado": datos_modelo.duplicated(["article_id", "week"]).sum(),
    "Valores vacíos": datos_modelo.isna().sum().sum(),
    "Valores numéricos no finitos": (~np.isfinite(numeros.to_numpy())).sum(),
    "Ventas negativas": (datos_modelo[["sales", "target_next_week_sales"]] < 0).any(axis=1).sum(),
    "Precios negativos": (datos_modelo[["min_price", "mean_price", "max_price"]] < 0).any(axis=1).sum(),
    "Media fuera del intervalo de precios": (
        (datos_modelo["mean_price"] < datos_modelo["min_price"] - tolerancia)
        | (datos_modelo["mean_price"] > datos_modelo["max_price"] + tolerancia)).sum(),
    "Proporción online incorrecta": (~datos_modelo["online_share"].between(0, 1)).sum(),
    "Ventas online mayores que totales": (datos_modelo["online_sales"] > datos_modelo["sales"]).sum(),
    "Objetivos desconocidos": (~datos_modelo["target_is_observed"]).sum(),
    "Historial desconocido": (~datos_modelo["history_is_observed"]).sum()
})
display(incidencias.to_frame("cantidad"))
if incidencias.any():
    raise ValueError("Revisa las incidencias antes de continuar")
display(datos_modelo[["sales", "mean_price", "target_next_week_sales"]].describe())

,tipo,vacios
week,datetime64[ns],0
article_id,string,0
sales,int64,0
min_price,float64,0
max_price,float64,0
online_sales,int64,0
mean_price,float64,0
online_share,float64,0
lag_1_sales,int64,0
lag_2_sales,int64,0


,cantidad
Artículo-semana duplicado,0
Valores vacíos,0
Valores numéricos no finitos,0
Ventas negativas,0
Precios negativos,0
Media fuera del intervalo de precios,0
Proporción online incorrecta,0
Ventas online mayores que totales,0
Objetivos desconocidos,0
Historial desconocido,0


,sales,mean_price,target_next_week_sales
count,1.887139e+06,1.887139e+06,1.887139e+06
mean,1.467708e+01,2.821217e-02,1.383003e+01
std,3.589778e+01,2.333422e-02,3.506646e+01
min,1.000000e+00,1.016949e-04,0.000000e+00
25%,1.000000e+00,1.354237e-02,1.000000e+00
50%,4.000000e+00,2.286441e-02,3.000000e+00
75%,1.200000e+01,3.388136e-02,1.200000e+01
max,3.100000e+03,5.067797e-01,3.100000e+03


## 3. Entender las fechas y los ceros

Solo se conservan semanas actuales con siete días cubiertos, doce semanas previas
completas y una semana siguiente completa. Que haya transacciones cada día no prueba
que el archivo contenga todas las ventas reales; la cobertura depende de la fuente.
El calendario usa `iso_year` e `iso_week`, sin mezclar año natural y semana ISO.

Si un artículo no tiene registros en una semana cubierta, sus ventas observadas son
cero. No interpretamos ese cero como prueba de que el artículo estaba disponible.

In [3]:
print("Transacciones de origen:", resumen["source_transactions"])
print("Primer día original:", resumen["first_date"])
print("Último día original:", resumen["last_date"])
print("Filas de semanas parciales excluidas:", resumen["partial_week_rows_excluded"])
print("Filas idénticas de transacciones conservadas:", resumen["identical_transaction_rows_preserved"])
print("Primera semana usada:", datos_modelo["week"].min().date())
print("Última semana usada:", datos_modelo["week"].max().date())
print("Última semana objetivo:", datos_modelo["target_week"].max().date())

# Comprobamos el horizonte y la coherencia ISO.
assert (datos_modelo["target_week"] == datos_modelo["week"] + pd.Timedelta(weeks=1)).all()
calendario = datos_modelo["week"].dt.isocalendar()
assert (datos_modelo["iso_year"] == calendario.year).all()
assert (datos_modelo["iso_week"] == calendario.week).all()

Transacciones de origen: 31788324
Primer día original: 2018-09-20
Último día original: 2020-09-22
Filas de semanas parciales excluidas: 29137
Filas idénticas de transacciones conservadas: 2974905
Primera semana usada: 2018-12-17
Última semana usada: 2020-09-07
Última semana objetivo: 2020-09-14


## 4. Revisar la separación temporal por porcentajes

Primero pedimos **70 % de filas para entrenamiento, 15 % para validación y 15 %
para prueba**. El procesador ordena las semanas objetivo y busca conjuntamente
las fechas que más se acercan a esas proporciones, sin partir ninguna semana.
Por eso los porcentajes reales pueden diferir ligeramente de los solicitados.
Se utilizan tamaños de semanas, nunca las ventas futuras, para elegir los cortes.

La columna `split` ya está guardada: aquí comprobamos y explicamos el reparto.
Las fechas de la tabla son semanas objetivo. Un artículo puede aparecer en varios
periodos porque evaluamos su futuro. Si se usa gap, se informa aparte y se excluye
del denominador de porcentajes. En este experimento semanal gap es cero.


In [4]:
# Leemos las proporciones pedidas y las fechas elegidas automáticamente.
configuracion_split = resumen["split_configuration"]
print("Porcentajes solicitados:", configuracion_split["requested_percentages"])
print("Primera semana objetivo de validación:", configuracion_split["validation_start"])
print("Primera semana objetivo de prueba:", configuracion_split["test_start"])
orden = ["train", "validation", "test"]
resumen_particiones = datos_modelo.groupby("split").agg(
    filas=("article_id", "size"),
    primer_objetivo=("target_week", "min"),
    ultimo_objetivo=("target_week", "max")
).reindex(orden)
if resumen_particiones.isna().any().any():
    raise ValueError("Alguna partición está vacía")

# El porcentaje se calcula sobre las filas utilizadas, excluyendo cualquier gap.
resumen_particiones["porcentaje"] = (
    100 * resumen_particiones["filas"] / resumen_particiones["filas"].sum()
).round(2)
resumen_particiones["porcentaje_solicitado"] = pd.Series(
    configuracion_split["requested_percentages"])
display(resumen_particiones)
print("Filas excluidas por separación adicional:", datos_modelo["split"].eq("gap").sum())
for anterior, posterior in [("train", "validation"), ("validation", "test")]:
    assert (resumen_particiones.loc[anterior, "ultimo_objetivo"]
            < resumen_particiones.loc[posterior, "primer_objetivo"])

Porcentajes solicitados: {'train': 70.0, 'validation': 15.0, 'test': 15.0}
Primera semana objetivo de validación: 2020-03-02
Primera semana objetivo de prueba: 2020-06-15


,filas,primer_objetivo,ultimo_objetivo,porcentaje,porcentaje_solicitado
split,,,,,
train,1326434,2018-12-24,2020-02-24,70.29,70.0
validation,271215,2020-03-02,2020-06-08,14.37,15.0
test,289490,2020-06-15,2020-09-14,15.34,15.0


Filas excluidas por separación adicional: 0


## 5. Predicción de referencia

Suponemos que las ventas de la semana siguiente serán iguales a las de la actual.
Medimos el error en validación; no consultamos las métricas de prueba hasta elegir
el modelo definitivo. MAE es el error absoluto medio y RMSE penaliza más los errores grandes.

In [5]:
# Todavía no entrenamos un modelo: calculamos una referencia que habrá que mejorar.
validacion = datos_modelo.loc[datos_modelo["split"] == "validation"]
ventas_reales = validacion["target_next_week_sales"]
predicciones = validacion["sales"]
metricas_referencia = pd.DataFrame([{
    "metodo": "Repetir las ventas de la semana actual", "particion": "validation",
    "MAE": mean_absolute_error(ventas_reales, predicciones),
    "RMSE": mean_squared_error(ventas_reales, predicciones) ** 0.5
}])
display(metricas_referencia.round(3))

,metodo,particion,MAE,RMSE
0,Repetir las ventas de la semana actual,validation,8.113,25.638


## Qué falta para entrenar

Preparar categorías y, cuando proceda, escalado usando exclusivamente entrenamiento;
comparar modelos en validación y evaluar una vez en prueba. Los pesos y umbrales
de clasificación también se aprenden únicamente con entrenamiento.

No utilices `target_next_week_sales`, `target_week`, `split`, `target_is_observed`
ni `history_is_observed` como variables predictoras. Conserva los identificadores
para unir y auditar datos, no como una variable numérica continua.

Los cambios se guardan con Ctrl + S. Reiniciar el kernel borra las variables de
memoria; Ejecutar todo reconstruye los resultados desde los archivos publicados.